<a href="https://colab.research.google.com/github/t0bleronee/BEE-102-Spring-2025-Assignment/blob/main/04_Vitierbi_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Q4: Writing Viterbi Algorithm for the Primer

**Parameters** (As defined in Nature Prime):

- states: List of states that represent biological components (Exon, 5′ Splice Site, Intron).

- startprobabilities: The probability of starting in the Exon state.

- transprobabilities: Transition probabilities between different states.

- emitprobabilities: Emission probabilities for observing specific nucleotides in each state.

- statetrace: The sequence of states corresponding to the observed nucleotide sequence.

- seqdata: The nucleotide sequence that is observed.


**Function:**

- get_log_prob_of_path_given_sequence(statetrace, seqdata): This function calculates and returns the logarithm of the probability of the given state sequence for the observed nucleotide sequence.

**Calculating the Log-Probability of a Known State Path**

To verify that our probabilistic model is correctly defined, we first compute the log-probability of a known sequence of states (also called a path) emitting a given DNA sequence. This involves:

- Iterating over the sequence and corresponding path:

  1. At each step, retrieve the transition probability from the previous state to the current state.

  2. Also retrieve the emission probability of the current nucleotide given the current state.

  3. Multiply the transition and emission probabilities, then take the logarithm of the result.

- Accumulate these log-values across all steps.

- If the path ends in an intron, include the log-probability of transitioning to the end state.

- The sum of all log-probabilities gives the total log-likelihood of the known state path producing the observed sequence.

- This process is essential for validating that the model probabilities (transition and emission) behave as expected.



In [ ]:
import numpy as np
import math

def safe_log(x):
    """Safe log calculation handling zero probability case."""
    return -math.inf if x == 0 else math.log(x)

def calculate_path_log_prob(state_path, sequence, start_probs, trans_probs, emit_probs):
    """Calculate log probability of a state path given a sequence."""
    if len(state_path) != len(sequence):
        raise ValueError("State path and sequence must have same length")

    log_prob = 0.0

    for i in range(len(sequence)):
        current_state = state_path[i]
        current_nuc = sequence[i]

        # Handle start probability for first position
        if i == 0:
            log_prob += safe_log(start_probs[current_state])
        else:
            log_prob += safe_log(trans_probs[state_path[i-1]][current_state])

        log_prob += safe_log(emit_probs[current_state][current_nuc])

    # Handle transition to end state if final state is I
    if state_path[-1] == 'I':
        log_prob += safe_log(trans_probs['I']['end'])

    return log_prob

In [ ]:
# Define HMM parameters
states = ['E', '5', 'I']

start_probabilities = {
    'E': 1.0,  # Always start in E state
    '5': 0.0,
    'I': 0.0
}

transition_probabilities = {
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'end': 0.1}
}

emission_probabilities = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

# Example sequence and path
example_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
example_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

# Calculate and print log probability
path_prob = calculate_path_log_prob(
    example_path,
    example_sequence,
    start_probabilities,
    transition_probabilities,
    emission_probabilities
)
print(f"Log probability of path: {round(path_prob, 4)}")

Log probability of path: -41.2197


## Steps involved in Viterbi algorithm

### 1. Initialize

- Set the initial probabilities based on the first observation.

### 2. Dynamic Programming Iteration:

For every position in the sequence:

- Evaluate all possible transitions from previous states.

- For each possibility, compute the total log-probability by adding:

  - The log-probability of reaching the previous state.

  - The log of the transition to the current state.

  - The log of the emission of the current nucleotide in the current state.

- Select the maximum among these and store it in the matrix.

- Also store a backpointer to trace the best previous state.

 **Backtracking:**

  - After filling in the matrix, locate the state in the last column with the highest log-probability.

  - Use the stored backpointers to trace back the most likely sequence of hidden states.

### 3. Terminate

-  At the last observation, select the state with the highest probability.


### 4. Output
- Trace back the most probable sequence of states and calculate its probability.
- The most likely hidden-state sequence  
- It's log-probability  


In [ ]:
def viterbi_decode(sequence, states, start_probs, trans_probs, emit_probs):
    """Viterbi algorithm for finding most probable state path."""
    # Initialize DP tables
    viterbi_table = [{}]
    path_history = {}

    # Initialization step
    for state in states:
        viterbi_table[0][state] = (
            safe_log(start_probs[state]) +
            safe_log(emit_probs[state].get(sequence[0], 0))
        )
        path_history[state] = [state]

    # Recursion step
    for t in range(1, len(sequence)):
        viterbi_table.append({})
        new_paths = {}

        for current_state in states:
            max_prob = -math.inf
            best_prev_state = None

            for prev_state in states:
                trans_prob = trans_probs.get(prev_state, {}).get(current_state, 0)

                if trans_prob > 0:
                    emit_prob = emit_probs[current_state].get(sequence[t], 0)

                    if emit_prob > 0:
                        current_prob = (
                            viterbi_table[t-1][prev_state] +
                            math.log(trans_prob) +
                            math.log(emit_prob)
                        )

                        if current_prob > max_prob:
                            max_prob = current_prob
                            best_prev_state = prev_state

            viterbi_table[t][current_state] = max_prob

            if best_prev_state is not None:
                new_paths[current_state] = path_history[best_prev_state] + [current_state]
            else:
                new_paths[current_state] = [current_state]

        path_history = new_paths

    # Termination step
    final_step = len(sequence) - 1
    best_state = max(states, key=lambda s: viterbi_table[final_step][s])

    return (
        round(viterbi_table[final_step][best_state], 2),
        ''.join(path_history[best_state])
    )

# Run Viterbi algorithm
best_log_prob, best_path = viterbi_decode(
    example_sequence,
    states,
    start_probabilities,
    transition_probabilities,
    emission_probabilities
)

print(f"Viterbi best log probability: {best_log_prob}")
print(f"Most probable path: {best_path}")

Viterbi best log probability: -38.68
Most probable path: EEEEEEEEEEEEEEEEEEEEEEEEEE
